In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim




In [2]:
import json
import os
from pathlib import Path
from torch.utils.data import TensorDataset

def load_lidar_ranges_dataset(base_path="cells_kernels", start_cell=0, end_cell=47):
    """
    Load lidar_scan.ranges from all JSON files in cells_kernels/c0 to c47/deg0 folders
    and return as a PyTorch TensorDataset.
    
    Args:
        base_path: Base path to cells_kernels directory
        start_cell: Starting cell number (default: 0)
        end_cell: Ending cell number (default: 47)
    
    Returns:
        TensorDataset: PyTorch dataset containing all lidar ranges
    """
    all_ranges = []
    
    # Iterate through all cell folders from c0 to c47
    for cell_num in range(start_cell, end_cell + 1):
        cell_folder = f"c{cell_num}"
        deg0_path = Path(base_path) / cell_folder / "deg0"
        
        # Check if deg0 folder exists
        if not deg0_path.exists():
            print(f"Warning: {deg0_path} does not exist, skipping...")
            continue
        
        # Find all JSON files in deg0 folder
        json_files = list(deg0_path.glob("*.json"))
        
        if len(json_files) == 0:
            print(f"Warning: No JSON files found in {deg0_path}")
            continue
        
        print(f"Processing {cell_folder}: Found {len(json_files)} JSON files")
        
        # Extract lidar_scan.ranges from each JSON file
        for json_file in json_files:
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                    
                # Extract lidar_scan.ranges
                if 'lidar_scan' in data and 'ranges' in data['lidar_scan']:
                    ranges = data['lidar_scan']['ranges']
                    all_ranges.append(ranges)
                else:
                    print(f"Warning: No lidar_scan.ranges found in {json_file}")
                    
            except Exception as e:
                print(f"Error reading {json_file}: {e}")
                continue
    
    if len(all_ranges) == 0:
        raise ValueError("No lidar ranges found in any JSON files!")
    
    # Convert to numpy array and then to torch tensor
    import numpy as np
    ranges_array = np.array(all_ranges, dtype=np.float32)
    ranges_tensor = torch.from_numpy(ranges_array)
    
    print(f"\nTotal samples loaded: {len(all_ranges)}")
    print(f"Shape of dataset: {ranges_tensor.shape}")
    
    # Create TensorDataset (since we only have features, no labels)
    # TensorDataset expects at least one tensor, we'll use the ranges as both X
    dataset = TensorDataset(ranges_tensor)
    
    return dataset, ranges_tensor

# Example usage:
dataset, ranges_tensor = load_lidar_ranges_dataset()


Processing c0: Found 100 JSON files
Processing c1: Found 100 JSON files
Processing c2: Found 100 JSON files
Processing c3: Found 100 JSON files
Processing c4: Found 100 JSON files
Processing c5: Found 100 JSON files
Processing c6: Found 100 JSON files
Processing c7: Found 100 JSON files
Processing c8: Found 100 JSON files
Processing c9: Found 100 JSON files
Processing c10: Found 100 JSON files
Processing c11: Found 100 JSON files
Processing c12: Found 100 JSON files
Processing c13: Found 100 JSON files
Processing c14: Found 16 JSON files
Processing c15: Found 100 JSON files
Processing c16: Found 100 JSON files
Processing c17: Found 100 JSON files
Processing c18: Found 100 JSON files
Processing c19: Found 100 JSON files
Processing c20: Found 100 JSON files
Processing c21: Found 100 JSON files
Processing c22: Found 100 JSON files
Processing c23: Found 100 JSON files
Processing c24: Found 100 JSON files
Processing c25: Found 100 JSON files
Processing c26: Found 100 JSON files
Processing c

In [7]:
# VAE Model Definition
class LidarVAE(nn.Module):
    def __init__(self, input_dim=640, hidden_dims=[512, 256, 128], latent_dim=32):
        """
        Variational Autoencoder for Lidar Scan Data
        
        Args:
            input_dim: Dimension of input lidar ranges (640)
            hidden_dims: List of hidden layer dimensions for encoder/decoder
            latent_dim: Dimension of latent space
        """
        super(LidarVAE, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder
        encoder_layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim)
            ])
            prev_dim = hidden_dim
        
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Latent space layers (mu and logvar)
        self.fc_mu = nn.Linear(prev_dim, latent_dim)
        self.fc_logvar = nn.Linear(prev_dim, latent_dim)
        
        # Decoder
        decoder_layers = []
        prev_dim = latent_dim
        for hidden_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim)
            ])
            prev_dim = hidden_dim
        
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        decoder_layers.append(nn.Sigmoid())  # Normalize output to [0, 1] range
        
        self.decoder = nn.Sequential(*decoder_layers)
    
    def encode(self, x):
        """Encode input to latent space parameters"""
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        """Decode latent vector to reconstruction"""
        return self.decoder(z)
    
    def forward(self, x):
        """Forward pass"""
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar


In [ ]:
# Loss function for VAE
def vae_loss(recon_x, x, mu, logvar, recon_loss_weight=1.0, kl_weight=1.0):
    """
    VAE Loss = Reconstruction Loss + KL Divergence
    
    Args:
        recon_x: Reconstructed input
        x: Original input
        mu: Mean of latent distribution
        logvar: Log variance of latent distribution
        recon_loss_weight: Weight for reconstruction loss
        kl_weight: Weight for KL divergence loss
    """
    # Reconstruction loss (MSE)
    recon_loss = nn.functional.mse_loss(recon_x, x, reduction='sum')
    
    # KL divergence loss: -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    # Total loss
    total_loss = recon_loss_weight * recon_loss + kl_weight * kl_loss
    
    return total_loss, recon_loss, kl_loss


In [ ]:
# Normalize data to [0, 1] range for better training
def normalize_data(data):
    """Normalize data to [0, 1] range"""
    data_min = data.min()
    data_max = data.max()
    normalized = (data - data_min) / (data_max - data_min + 1e-8)
    return normalized, data_min, data_max

def denormalize_data(normalized_data, data_min, data_max):
    """Denormalize data back to original range"""
    return normalized_data * (data_max - data_min) + data_min

# Normalize the dataset
ranges_normalized, data_min, data_max = normalize_data(ranges_tensor)
print(f"Data range: [{data_min:.4f}, {data_max:.4f}]")
print(f"Normalized range: [{ranges_normalized.min():.4f}, {ranges_normalized.max():.4f}]")

# Create normalized dataset
normalized_dataset = TensorDataset(ranges_normalized)


In [ ]:
# Training setup
from torch.utils.data import DataLoader, random_split

# Hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
NUM_EPOCHS = 50
LATENT_DIM = 32
HIDDEN_DIMS = [512, 256, 128]
RECON_LOSS_WEIGHT = 1.0
KL_WEIGHT = 0.0001  # Start with small KL weight, can increase during training

# Split dataset into train and validation
train_size = int(0.8 * len(normalized_dataset))
val_size = len(normalized_dataset) - train_size
train_dataset, val_dataset = random_split(normalized_dataset, [train_size, val_size])

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {train_size}, Validation samples: {val_size}")

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = LidarVAE(
    input_dim=640,
    hidden_dims=HIDDEN_DIMS,
    latent_dim=LATENT_DIM
).to(device)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# Training loop
def train_epoch(model, train_loader, optimizer, device, kl_weight):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    total_recon_loss = 0
    total_kl_loss = 0
    
    for batch_idx, (data,) in enumerate(train_loader):
        data = data.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data)
        
        # Calculate loss
        loss, recon_loss, kl_loss = vae_loss(
            recon_batch, data, mu, logvar,
            recon_loss_weight=1.0,
            kl_weight=kl_weight
        )
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Accumulate losses
        total_loss += loss.item()
        total_recon_loss += recon_loss.item()
        total_kl_loss += kl_loss.item()
    
    return total_loss / len(train_loader.dataset), \
           total_recon_loss / len(train_loader.dataset), \
           total_kl_loss / len(train_loader.dataset)

def validate(model, val_loader, device, kl_weight):
    """Validate the model"""
    model.eval()
    total_loss = 0
    total_recon_loss = 0
    total_kl_loss = 0
    
    with torch.no_grad():
        for data, in val_loader:
            data = data.to(device)
            recon_batch, mu, logvar = model(data)
            
            loss, recon_loss, kl_loss = vae_loss(
                recon_batch, data, mu, logvar,
                recon_loss_weight=1.0,
                kl_weight=kl_weight
            )
            
            total_loss += loss.item()
            total_recon_loss += recon_loss.item()
            total_kl_loss += kl_loss.item()
    
    return total_loss / len(val_loader.dataset), \
           total_recon_loss / len(val_loader.dataset), \
           total_kl_loss / len(val_loader.dataset)

# Training history
train_losses = []
val_losses = []
train_recon_losses = []
val_recon_losses = []
train_kl_losses = []
val_kl_losses = []

print("Starting training...\n")
for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_recon, train_kl = train_epoch(model, train_loader, optimizer, device, KL_WEIGHT)
    
    # Validate
    val_loss, val_recon, val_kl = validate(model, val_loader, device, KL_WEIGHT)
    
    # Update learning rate
    scheduler.step(val_loss)
    
    # Store history
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_recon_losses.append(train_recon)
    val_recon_losses.append(val_recon)
    train_kl_losses.append(train_kl)
    val_kl_losses.append(val_kl)
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]')
        print(f'  Train Loss: {train_loss:.4f} (Recon: {train_recon:.4f}, KL: {train_kl:.4f})')
        print(f'  Val Loss: {val_loss:.4f} (Recon: {val_recon:.4f}, KL: {val_kl:.4f})')
        print()

print("Training completed!")


In [ ]:
# Plot training history
plt.figure(figsize=(15, 5))

# Total loss
plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Total Loss')
plt.legend()
plt.grid(True)

# Reconstruction loss
plt.subplot(1, 3, 2)
plt.plot(train_recon_losses, label='Train Recon Loss')
plt.plot(val_recon_losses, label='Val Recon Loss')
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Loss')
plt.title('Reconstruction Loss')
plt.legend()
plt.grid(True)

# KL divergence loss
plt.subplot(1, 3, 3)
plt.plot(train_kl_losses, label='Train KL Loss')
plt.plot(val_kl_losses, label='Val KL Loss')
plt.xlabel('Epoch')
plt.ylabel('KL Divergence Loss')
plt.title('KL Divergence Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Visualize reconstructions
model.eval()
with torch.no_grad():
    # Get a batch of validation data
    sample_batch = next(iter(val_loader))[0].to(device)
    recon_batch, mu, logvar = model(sample_batch)
    
    # Denormalize
    sample_batch_denorm = denormalize_data(sample_batch.cpu(), data_min, data_max)
    recon_batch_denorm = denormalize_data(recon_batch.cpu(), data_min, data_max)
    
    # Plot some examples
    num_samples = 5
    fig, axes = plt.subplots(2, num_samples, figsize=(20, 6))
    
    for i in range(num_samples):
        # Original
        axes[0, i].plot(sample_batch_denorm[i].numpy())
        axes[0, i].set_title(f'Original Sample {i+1}')
        axes[0, i].set_xlabel('Lidar Index')
        axes[0, i].set_ylabel('Range')
        axes[0, i].grid(True)
        
        # Reconstructed
        axes[1, i].plot(recon_batch_denorm[i].numpy())
        axes[1, i].set_title(f'Reconstructed Sample {i+1}')
        axes[1, i].set_xlabel('Lidar Index')
        axes[1, i].set_ylabel('Range')
        axes[1, i].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate reconstruction error
    mse_per_sample = torch.mean((sample_batch_denorm - recon_batch_denorm) ** 2, dim=1)
    print(f"Average MSE per sample: {mse_per_sample.mean():.6f}")
    print(f"MSE std: {mse_per_sample.std():.6f}")


In [ ]:
# Generate new samples from latent space
model.eval()
with torch.no_grad():
    # Sample from prior distribution (standard normal)
    num_generated = 5
    z_samples = torch.randn(num_generated, LATENT_DIM).to(device)
    generated_samples = model.decode(z_samples)
    
    # Denormalize
    generated_samples_denorm = denormalize_data(generated_samples.cpu(), data_min, data_max)
    
    # Plot generated samples
    fig, axes = plt.subplots(1, num_generated, figsize=(20, 3))
    
    for i in range(num_generated):
        axes[i].plot(generated_samples_denorm[i].numpy())
        axes[i].set_title(f'Generated Sample {i+1}')
        axes[i].set_xlabel('Lidar Index')
        axes[i].set_ylabel('Range')
        axes[i].grid(True)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Save the trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'data_min': data_min,
    'data_max': data_max,
    'latent_dim': LATENT_DIM,
    'hidden_dims': HIDDEN_DIMS,
    'input_dim': 640,
    'train_losses': train_losses,
    'val_losses': val_losses,
}, 'lidar_vae_model.pth')

print("Model saved to 'lidar_vae_model.pth'")


In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load("/home/mehdi/NerualRateMaps/lidar_vae_model.pth", map_location=device)  # keep weights_only default

# Re-create model with same architecture params
model = LidarVAE(
    input_dim=checkpoint["input_dim"],
    latent_dim=checkpoint["latent_dim"],
    hidden_dims=checkpoint["hidden_dims"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

/tmp/ipykernel_24714/2295992088.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("/home/mehdi/NerualRateMaps/lidar_vae_model.pth", map_location=de

LidarVAE(
  (encoder): Sequential(
    (0): Linear(in_features=640, out_features=512, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): ReLU()
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): Linear(in_features=256, out_features=128, bias=True)
    (7): ReLU()
    (8): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (fc_mu): Linear(in_features=128, out_features=32, bias=True)
  (fc_logvar): Linear(in_features=128, out_features=32, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Linear(in_features=128, out_features=256, bias=True)
    (4): ReLU()
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine

In [ ]:
# Function to plot vector field from JSON data
def plot_vector_field_from_json(json_path='trj/vector_field_data.json', 
                                 cell_ls=None, 
                                 output_path='trj/vector_field_plot.png',
                                 show_cells=True,
                                 scale_arrows=True):
    """
    Plot vector field from lidar data saved in JSON format.
    
    Args:
        json_path: Path to JSON file containing vector field data
        cell_ls: List of cell objects for plotting boundaries (optional)
        output_path: Path to save the output plot
        show_cells: Whether to plot cell boundaries
        scale_arrows: Whether to scale arrows based on magnitude
    
    Returns:
        fig, ax: Matplotlib figure and axes objects
    """
    import json
    import os
    from pathlib import Path
    
    # Load JSON data
    json_path = Path(json_path)
    if not json_path.exists():
        raise FileNotFoundError(f"JSON file not found: {json_path}")
    
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    if len(data) == 0:
        raise ValueError("JSON file is empty")
    
    print(f"Loaded {len(data)} data points from {json_path}")
    
    # Extract positions and control vectors
    positions = []
    u_vectors = []
    headings = []
    lidar_ranges = []
    
    for entry in data:
        pos = entry.get('position', [0, 0])
        u = entry.get('u', [0, 0])
        heading = entry.get('heading', 0)
        lidar = entry.get('lidar_ranges', [])
        
        positions.append(pos)
        u_vectors.append(u)
        headings.append(heading)
        if lidar:
            lidar_ranges.append(lidar)
    
    # Convert to numpy arrays
    positions = np.array(positions)
    u_vectors = np.array(u_vectors)
    headings = np.array(headings)
    
    print(f"Position range: x=[{positions[:, 0].min():.3f}, {positions[:, 0].max():.3f}], "
          f"y=[{positions[:, 1].min():.3f}, {positions[:, 1].max():.3f}]")
    print(f"Control vector range: u_x=[{u_vectors[:, 0].min():.3f}, {u_vectors[:, 0].max():.3f}], "
          f"u_y=[{u_vectors[:, 1].min():.3f}, {u_vectors[:, 1].max():.3f}]")
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Plot cell boundaries if provided
    if show_cells and cell_ls is not None:
        try:
            from cell_configs import cell_ls as default_cell_ls
            if cell_ls is None:
                cell_ls = default_cell_ls
            
            # Plot cell boundaries
            for cell in cell_ls:
                vrt = np.array(cell.vrt)
                for i in range(len(vrt) - 1):
                    ax.plot([vrt[i, 0], vrt[i+1, 0]], 
                           [vrt[i, 1], vrt[i+1, 1]], 
                           color='gray', linewidth=1, alpha=0.7)
                ax.plot([vrt[0, 0], vrt[-1, 0]], 
                       [vrt[0, 1], vrt[-1, 1]], 
                       color='gray', linewidth=1, alpha=0.7)
            
            print("Cell boundaries plotted")
        except Exception as e:
            print(f"Warning: Could not plot cell boundaries: {e}")
    
    # Calculate arrow scaling if needed
    if scale_arrows:
        magnitudes = np.sqrt(u_vectors[:, 0]**2 + u_vectors[:, 1]**2)
        max_magnitude = np.max(magnitudes)
        if max_magnitude > 0:
            # Scale so arrows are visible but not too large
            scale = max_magnitude * 3.0
        else:
            scale = 1.0
    else:
        scale = None
    
    # Plot vector field
    ax.quiver(positions[:, 0], positions[:, 1], 
              u_vectors[:, 0], u_vectors[:, 1],
              angles='xy', scale_units='xy', 
              scale=scale,
              width=0.003,
              alpha=0.7,
              color='blue')
    
    # Add labels and formatting
    ax.set_xlabel('X Position (m)', fontsize=12)
    ax.set_ylabel('Y Position (m)', fontsize=12)
    ax.set_title('Vector Field from Lidar Data', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    # Save the plot
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Vector field plot saved to: {output_path}")
    
    plt.show()
    
    return fig, ax

# Example usage:
plot_vector_field_from_json('trj/vector_field_data.json')


In [ ]:
# Plot vector field from JSON data
plot_vector_field_from_json('trj/vector_field_data.json', 
                            output_path='trj/vector_field_plot.png',
                            show_cells=True,
                            scale_arrows=True)


In [ ]:
# Function to plot vector field from lidar data using VAE encoding
def plot_vector_field_from_lidar(json_path='trj/vector_field_data.json',
                                 vae_model_path='lidar_vae_model.pth',
                                 output_path='trj/vector_field_plot_from_lidar.png',
                                 show_cells=True,
                                 device='cpu'):
    """
    Plot vector field from lidar data by:
    1. Loading lidar data from JSON
    2. Encoding lidar ranges with VAE
    3. Computing control vectors using controller (similar to gazebo_neural_analysis.py)
    4. Plotting the vector field
    
    Args:
        json_path: Path to JSON file containing lidar data
        vae_model_path: Path to VAE model file
        output_path: Path to save the output plot
        show_cells: Whether to plot cell boundaries
        device: Device to run VAE on ('cpu' or 'cuda')
    
    Returns:
        fig, ax: Matplotlib figure and axes objects
    """
    import json
    from pathlib import Path
    from find_controller_orientation import control_gain_load
    from cell_configs import cell_ls
    
    # Load JSON data
    json_path = Path(json_path)
    if not json_path.exists():
        raise FileNotFoundError(f"JSON file not found: {json_path}")
    
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    if len(data) == 0:
        raise ValueError("JSON file is empty")
    
    print(f"Loaded {len(data)} data points from {json_path}")
    
    # Load VAE model
    vae_model_path = Path(vae_model_path)
    if not vae_model_path.exists():
        raise FileNotFoundError(f"VAE model file not found: {vae_model_path}")
    
    print(f"Loading VAE model from {vae_model_path}...")
    VAE, vae_data_min, vae_data_max = load_vae_model(model_path=str(vae_model_path), device=device)
    vae_device = next(VAE.parameters()).device
    print(f"VAE model loaded successfully. Device: {vae_device}")
    
    # Initialize control gain loader
    control_gain_loader = control_gain_load()
    
    # Helper function to find cell
    def find_cell(position, cell_ls):
        ls_flag = []
        for i in range(len(cell_ls)):
            ls_flag.append(cell_ls[i].check_in_polygon(np.reshape(position, (1, 2))))
        cell_indices = [i for i, x in enumerate(ls_flag) if x]
        if cell_indices:
            return cell_indices[0]
        else:
            print(f"Warning: Position {position} not in any defined cell, using cell 0")
            return 0
    
    # Helper function to encode lidar with VAE
    def encode_lidar_with_vae(VAE, lidar_ranges, vae_data_min, vae_data_max, vae_device):
        if not isinstance(lidar_ranges, np.ndarray):
            lidar_ranges = np.array(lidar_ranges)
        
        single_sample = len(lidar_ranges.shape) == 1
        if single_sample:
            lidar_ranges = lidar_ranges.reshape(1, -1)
        
        if isinstance(lidar_ranges, torch.Tensor):
            lidar_ranges = lidar_ranges.cpu().numpy()
        lidar_ranges = np.array(lidar_ranges, dtype=np.float32)
        
        data_min = float(vae_data_min)
        data_max = float(vae_data_max)
        lidar_normalized = (lidar_ranges - data_min) / (data_max - data_min + 1e-8)
        lidar_tensor = torch.from_numpy(lidar_normalized).float().to(vae_device)
        
        with torch.no_grad():
            mu, logvar = VAE.encode(lidar_tensor)
            z = mu
        
        z_numpy = z.cpu().numpy()
        return z_numpy[0] if single_sample else z_numpy
    
    # Extract data and compute control vectors
    positions = []
    u_vectors = []
    headings = []
    
    print("Computing control vectors from lidar data...")
    for i, entry in enumerate(data):
        pos = entry.get('position', [0, 0])
        heading = entry.get('heading', 0)
        lidar_ranges = entry.get('lidar_ranges', [])
        
        if not lidar_ranges or len(lidar_ranges) == 0:
            print(f"Warning: No lidar data for entry {i}, skipping")
            continue
        
        try:
            # Find current cell
            current_cell = find_cell(pos, cell_ls)
            
            # Load control gains
            K, Kb = control_gain_loader.interpolate_contorlgains(current_cell, heading)
            
            # Encode lidar with VAE
            measurement = encode_lidar_with_vae(VAE, lidar_ranges, 
                                               vae_data_min, vae_data_max, 
                                               vae_device)
            
            if measurement.ndim == 1:
                measurement = measurement.reshape(-1, 1)
            
            # Calculate control input: u = K @ measurement + Kb
            u = K @ measurement + Kb
            
            # Normalize and scale (similar to gazebo_neural_analysis.py)
            speed = 3.0
            u_norm = np.linalg.norm(u)
            if u_norm > 1e-8:
                u_normalized = u / u_norm
                u_scaled = u_normalized * speed
            else:
                u_scaled = u
            
            positions.append(pos)
            u_vectors.append(u_scaled.flatten())
            headings.append(heading)
            
            if (i + 1) % 10 == 0:
                print(f"Processed {i + 1}/{len(data)} entries...")
                
        except Exception as e:
            print(f"Error processing entry {i}: {e}")
            continue
    
    if len(positions) == 0:
        raise ValueError("No valid control vectors computed")
    
    # Convert to numpy arrays
    positions = np.array(positions)
    u_vectors = np.array(u_vectors)
    headings = np.array(headings)
    
    print(f"\nComputed {len(positions)} control vectors")
    print(f"Position range: x=[{positions[:, 0].min():.3f}, {positions[:, 0].max():.3f}], "
          f"y=[{positions[:, 1].min():.3f}, {positions[:, 1].max():.3f}]")
    print(f"Control vector range: u_x=[{u_vectors[:, 0].min():.3f}, {u_vectors[:, 0].max():.3f}], "
          f"u_y=[{u_vectors[:, 1].min():.3f}, {u_vectors[:, 1].max():.3f}]")
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Plot cell boundaries if requested
    if show_cells:
        try:
            for cell in cell_ls:
                vrt = np.array(cell.vrt)
                for i in range(len(vrt) - 1):
                    ax.plot([vrt[i, 0], vrt[i+1, 0]], 
                           [vrt[i, 1], vrt[i+1, 1]], 
                           color='gray', linewidth=1, alpha=0.7)
                ax.plot([vrt[0, 0], vrt[-1, 0]], 
                       [vrt[0, 1], vrt[-1, 1]], 
                       color='gray', linewidth=1, alpha=0.7)
                
                for barrier in cell.bar:
                    ax.plot([barrier[0][0], barrier[1][0]], 
                           [barrier[0][1], barrier[1][1]], 
                           color='red', linewidth=2, alpha=0.8)
                
                exit_vrt = cell.exit_vrt
                ax.plot([exit_vrt[0][0], exit_vrt[1][0]], 
                       [exit_vrt[0][1], exit_vrt[1][1]], 
                       color='green', linewidth=2, alpha=0.8)
            
            print("Cell boundaries plotted")
        except Exception as e:
            print(f"Warning: Could not plot cell boundaries: {e}")
    
    # Plot vector field
    quiver = ax.quiver(positions[:, 0], positions[:, 1], 
                       u_vectors[:, 0], u_vectors[:, 1],
                       angles='xy', scale_units='xy', 
                       scale=None,
                       width=0.003,
                       alpha=0.7,
                       color='blue',
                       headwidth=3,
                       headlength=4)
    
    ax.set_xlabel('X Position (m)', fontsize=12)
    ax.set_ylabel('Y Position (m)', fontsize=12)
    ax.set_title('Vector Field from Lidar Data (VAE Encoded)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    # Save the plot
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"\nVector field plot saved to: {output_path}")
    
    plt.show()
    
    return fig, ax

# Example usage:
# plot_vector_field_from_lidar('trj/vector_field_data.json')
